[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/90_exercises.ipynb)

# Bài tập — Chương 35: Reinforcement Learning (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import math

import numpy as np
from scipy.stats import poisson

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.envs import CartPole, CliffWalking, GridWorld

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

torch.set_num_threads(1)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Return và hệ số chiết khấu. (b) Phương trình Bellman kỳ vọng và tối ưu. (c) MC và TD(0) khác nhau thế nào (mục tiêu, bias, variance)? (d) SARSA vs Q-learning; on-policy vs off-policy. (e) Hai thủ thuật chính của DQN. (f) Định lý policy gradient và vai trò của baseline.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một sàn thương mại điện tử dùng Q-learning để **định giá động** trực tiếp trên khách hàng thật: ε = 0.3 (30% thời gian chọn giá ngẫu nhiên), phần thưởng = doanh thu trong ngày, γ = 0. Họ huấn luyện trên một bộ mô phỏng tự xây, báo cáo đường cong phần thưởng của **một** lần chạy trên chính bộ mô phỏng đó, rồi đưa lên production. Chỉ ra các vấn đề.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** MDP 2 trạng thái với chính sách cố định: từ A, nhận thưởng 1 rồi sang B; từ B, nhận thưởng 0 rồi với xác suất 0.5 về A, 0.5 kết thúc. γ = 0.9. Giải hệ Bellman $V_A = 1 + 0.9V_B$, $V_B = 0 + 0.9\cdot 0.5\,V_A$ → `V_A`, `V_B`. Tính thêm return `G` của chuỗi thưởng $[1, 0, 1, 0]$ với γ = 0.9.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
V_A = None
V_B = None
G = None
### END CODE HERE ###
print(f"V_A = {V_A:.4f}, V_B = {V_B:.4f}, G = {G:.4f}")

In [ ]:
sol = np.linalg.solve(np.array([[1, -0.9], [-0.45, 1]]), np.array([1.0, 0.0]))
check_close([V_A, V_B], sol, message="khớp nghiệm hệ tuyến tính")
check_close(G, 1.81, message="G = 1 + 0.81")

**Câu 4.** $Q(s, \cdot) = [2, 5]$, $Q(s', \cdot) = [3, 1]$. Tác tử ở $s$ làm hành động 0, nhận $r = 1$, sang $s'$, rồi (theo ε-greedy) **sẽ** làm hành động 1. α = 0.5, γ = 1. Tính giá trị mới $Q(s, 0)$ theo SARSA (`q_sarsa`) và theo Q-learning (`q_qlearning`).

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
q_sarsa = None
q_qlearning = None
### END CODE HERE ###
print(q_sarsa, q_qlearning)

In [ ]:
check_close(q_sarsa, 2.0, message="SARSA: mục tiêu 1 + Q(s', 1) = 2")
check_close(q_qlearning, 3.0, message="Q-learning: mục tiêu 1 + max Q(s', ·) = 4")

## L3 — Code

**Bài 5.** Viết `value_iteration(env, gamma, tol)` tổng quát (dùng `env.P`) trả về `(V, policy)`. Kiểm tra trên FrozenLake 8×8 trơn: (a) **phần dư Bellman** $\max_s|V(s) - \max_a\sum P[r + \gamma V(s')]|$ gần 0; (b) đánh giá chính xác chính sách tìm được (giải hệ tuyến tính) trùng $V$.

In [ ]:
def value_iteration(env, gamma=0.99, tol=1e-10):
    ### START CODE HERE ### (≈ 9 dòng)
    return None, None
    ### END CODE HERE ###

In [ ]:
env5 = GridWorld("8x8", slip=2 / 3)
V5, pi5 = value_iteration(env5)
Q5 = np.array([[sum(p * (r + 0.99 * V5[s2] * (not d)) for p, s2, r, d in env5.P[s][a]) for a in range(4)] for s in range(64)])
check_less(np.abs(V5 - Q5.max(1)).max(), 1e-8, "phần dư Bellman ≈ 0")
R5, P5 = np.zeros(64), np.zeros((64, 64))
for s in range(64):
    for p, s2, r, d in env5.P[s][pi5[s]]:
        R5[s] += p * r
        if not d:
            P5[s, s2] += p
check_close(np.linalg.solve(np.eye(64) - 0.99 * P5, R5), V5, tol=1e-7, message="đánh giá chính xác chính sách = V")
print(f"V*(S) trên 8×8 trơn = {V5[0]:.4f}")

**Bài 6.** **Expected SARSA**: mục tiêu là **kỳ vọng** theo chính sách ε-greedy thay vì một mẫu $a'$: $r + \gamma\sum_{a'}\pi(a'|s')Q(s', a')$. Viết `expected_target(Q_next, eps)` (Q_next: vector $Q(s', \cdot)$; hoà giữa các hành động tốt nhất thì chia đều phần tham lam), rồi so sánh tổng thưởng khi học trên Cliff Walking với SARSA và Q-learning (α = 0.5, ε = 0.1, 500 lượt, 20 seed).

In [ ]:
def expected_target(Q_next, eps):
    ### START CODE HERE ### (≈ 4 dòng)
    return None
    ### END CODE HERE ###


def cliff_control(method, seed, eps=0.1, alpha=0.5, episodes=500):
    env, rng, Q, returns = CliffWalking(), np.random.default_rng(seed), np.zeros((48, 4)), []
    act = lambda s: int(rng.integers(4)) if rng.random() < eps else int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))
    for _ in range(episodes):
        s, done, total = env.reset(), False, 0.0
        a = act(s)
        while not done:
            s2, r, done = env.step(a)
            total += r
            a2 = act(s2)
            if done:
                target = r
            elif method == "SARSA":
                target = r + Q[s2, a2]
            elif method == "Q-learning":
                target = r + Q[s2].max()
            else:
                target = r + expected_target(Q[s2], eps)
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
        returns.append(total)
    return np.mean(returns[-100:])


online6 = {m: np.mean([cliff_control(m, seed) for seed in range(20)]) for m in ["SARSA", "Q-learning", "Expected SARSA"]}
for m, v in online6.items():
    print(f"{m:15s}: tổng thưởng khi học (100 lượt cuối, TB 20 seed) = {v:.1f}")

In [ ]:
check_close(expected_target(np.array([1.0, 3.0, 3.0, 0.0]), 0.2), 0.05 * 1 + 0.45 * 3 + 0.45 * 3 + 0.05 * 0, message="hoà giữa hai hành động tốt nhất")
check_close(expected_target(np.array([1.0, 2.0]), 0.0), 2.0, message="ε = 0 → max")
check_greater(online6["Expected SARSA"], online6["SARSA"], "Expected SARSA ≥ SARSA (bớt nhiễu do lấy mẫu a')")
check_greater(online6["SARSA"], online6["Q-learning"], "SARSA an toàn hơn Q-learning khi đang học")

## L4 — Debug

**Bài 7.** Một cài đặt REINFORCE + baseline (cho sẵn) **không học được** — return còn thấp hơn chính sách ngẫu nhiên. Lỗi nằm ở hàm tính "return từ mỗi bước". Sửa `returns_to_go(rewards, gamma)` và so sánh (400 lượt).

In [ ]:
def returns_to_go_buggy(rewards, gamma):
    discounts = gamma ** np.arange(len(rewards))
    return np.cumsum(np.array(rewards) * discounts)                    # tổng thưởng (chiết khấu) TỪ ĐẦU lượt tới t


def returns_to_go(rewards, gamma):
    ### START CODE HERE ### (≈ 5 dòng)
    return returns_to_go_buggy(rewards, gamma)
    ### END CODE HERE ###


def reinforce(return_fn, episodes=400, seed=0):
    torch.manual_seed(seed)
    env, g = CartPole(), torch.Generator().manual_seed(seed)
    env.reset(seed=seed)
    policy = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2))
    value = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 1))
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=3e-3)
    lengths = []
    for _ in range(episodes):
        s, done, S, A, R = env.reset(), False, [], [], []
        while not done:
            with torch.no_grad():
                a = torch.multinomial(F.softmax(policy(torch.tensor(s, dtype=torch.float32)), -1), 1, generator=g).item()
            s2, r, done = env.step(a)
            S.append(s)
            A.append(a)
            R.append(r)
            s = s2
        St = torch.tensor(np.array(S), dtype=torch.float32)
        Gt = torch.tensor(return_fn(R, 0.99), dtype=torch.float32)
        log_probs = F.log_softmax(policy(St), -1)[torch.arange(len(A)), torch.tensor(A)]
        v = value(St).squeeze(1)
        loss = -(log_probs * (Gt - v).detach()).mean() + 0.005 * F.mse_loss(v, Gt)
        opt.zero_grad()
        loss.backward()
        opt.step()
        lengths.append(len(R))
    return np.mean(lengths[-100:])


results7 = {name: reinforce(fn) for name, fn in [("lỗi", returns_to_go_buggy), ("đã sửa", returns_to_go)]}
for name, v in results7.items():
    print(f"{name:7s}: return TB 100 lượt cuối = {v:.0f} (chính sách ngẫu nhiên ≈ 22)")

In [ ]:
check_close(returns_to_go([1, 1, 1], 0.5), [1.75, 1.5, 1.0], message="return từ mỗi bước")
check_less(results7["lỗi"], 60, "tái hiện: bản lỗi không học được")
check_greater(results7["đã sửa"], 300, "bản đã sửa học được")

## L5 — Phân tích

**Bài 8.** **On-policy vs off-policy khi thay đổi ε.** Trên Cliff Walking (α = 0.5, 1 000 lượt, 10 seed), với ε ∈ {0.01, 0.1, 0.3} chạy SARSA và Q-learning; lưu `online8[(method, ε)]` = tổng thưởng TB 100 lượt cuối khi học, và `path8[(method, ε)]` = danh sách độ dài đường đi của chính sách **tham lam** cuối cùng (None nếu chính sách tham lam không tới đích trong 60 bước). Nhận xét.

In [ ]:
def cliff_run(method, eps, seed, episodes=1000, alpha=0.5):
    env, rng, Q, returns = CliffWalking(), np.random.default_rng(seed), np.zeros((48, 4)), []
    act = lambda s: int(rng.integers(4)) if rng.random() < eps else int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))
    for _ in range(episodes):
        s, done, total = env.reset(), False, 0.0
        a = act(s)
        while not done:
            s2, r, done = env.step(a)
            total += r
            a2 = act(s2)
            target = r if done else r + (Q[s2, a2] if method == "SARSA" else Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
        returns.append(total)
    env, path_len = CliffWalking(), None
    s, done = env.reset(), False
    for step in range(1, 61):
        s, r, done = env.step(int(Q[s].argmax()))
        if done:
            path_len = step
            break
    return np.mean(returns[-100:]), path_len


online8, path8 = {}, {}
### START CODE HERE ### (≈ 5 dòng)
# với ε và method: chạy 10 seed bằng cliff_run, lưu trung bình tổng thưởng và danh sách độ dài đường tham lam
pass
### END CODE HERE ###

In [ ]:
check_true(all(p == 13 for eps in [0.01, 0.1, 0.3] for p in path8[("Q-learning", eps)]), "Q-learning: đường tham lam luôn tối ưu (13 bước) với mọi ε")
check_greater(online8[("Q-learning", 0.01)], online8[("Q-learning", 0.3)] + 100, "Q-learning: khám phá nhiều → trả giá đắt khi đang học")
check_greater(online8[("SARSA", 0.3)], online8[("Q-learning", 0.3)], "SARSA tốt hơn khi đang học với ε lớn")

## L6 — Thực tế

**Bài 9.** **Quản lý tồn kho bằng MDP.** Mỗi ngày: tồn kho $s \in \{0..10\}$; chọn đặt $a$ đơn vị (tối đa tới sức chứa 10, hàng về ngay); nhu cầu $D \sim$ Poisson(3); bán $\min(s + a, D)$; thưởng = 10·bán − 6·đặt − 1·tồn cuối ngày − 4·(nhu cầu không đáp ứng). γ = 0.95. (1) Xây mô hình `P[s][a]` = danh sách (xác suất, s', r) (gộp nhu cầu ≥ 29 vào một mức); (2) value iteration → `policy9[s]`; (3) mô phỏng 100 000 ngày để so sánh thưởng trung bình mỗi ngày của chính sách tối ưu với "đặt 3 mỗi ngày" và "luôn đặt đầy kho".

In [ ]:
MAX_STOCK, PRICE, ORDER_COST, HOLDING, LOST, GAMMA9 = 10, 10.0, 6.0, 1.0, 4.0, 0.95
demand_pmf = poisson(3).pmf(np.arange(30))
demand_pmf[-1] += 1 - demand_pmf.sum()
### START CODE HERE ### (≈ 20 dòng)
P9 = {}; policy9 = None
### END CODE HERE ###
print("lượng đặt tối ưu theo tồn kho 0..10:", policy9.tolist(), "| tồn sau khi đặt:", (np.arange(11) + policy9).tolist())


def simulate(policy, days=100_000, seed=0):
    rng, s, total = np.random.default_rng(seed), 0, 0.0
    for _ in range(days):
        a = policy(s)
        stock = min(s + a, MAX_STOCK)
        d = rng.poisson(3)
        sold = min(stock, d)
        total += PRICE * sold - ORDER_COST * a - HOLDING * (stock - sold) - LOST * (d - sold)
        s = stock - sold
    return total / days


rewards9 = {"tối ưu (value iteration)": simulate(lambda s: policy9[s]), "đặt 3 mỗi ngày": simulate(lambda s: min(3, MAX_STOCK - s)),
            "luôn đặt đầy kho": simulate(lambda s: MAX_STOCK - s)}
for k, v in rewards9.items():
    print(f"{k:25s}: thưởng trung bình mỗi ngày = {v:.3f}")

In [ ]:
check_close(sum(p for p, _, _ in P9[4][2]), 1.0, message="xác suất chuyển trạng thái cộng lại = 1")
order_up_to = np.arange(11) + policy9
check_true(len(set(order_up_to[:5])) == 1 and all(policy9[order_up_to[0]:] == 0), "chính sách có dạng 'đặt lên tới mức S' (base-stock)")
check_greater(rewards9["tối ưu (value iteration)"], max(rewards9["đặt 3 mỗi ngày"], rewards9["luôn đặt đầy kho"]) + 0.5, "tối ưu vượt các heuristic")

## L7 — Nâng cao

**Bài 10.** **Đánh giá chính sách off-policy (OPE).** Trong thực tế (gợi ý, quảng cáo, y tế) ta có **log** của chính sách cũ và muốn biết chính sách mới tốt đến đâu **trước khi** triển khai. Contextual bandit: ngữ cảnh $x \in \mathbb R^5$, 4 hành động, xác suất click $p(x, a)$ (cho sẵn — dùng để tính giá trị thật). Chính sách log: softmax ngẫu nhiên đã biết xác suất (propensity); chính sách mới: tham lam theo một mô hình tốt hơn. Với 50 bộ log (mỗi bộ 3 000 dòng), ước lượng giá trị chính sách mới bằng: **naive** (trung bình thưởng trên các dòng mà hành động log trùng hành động mới), **IPS** $\frac1n\sum\frac{\mathbb 1[a_i = \pi(x_i)]}{\mu(a_i|x_i)}r_i$, **SNIPS** (chia cho tổng trọng số thay vì $n$). Lưu `estimates10[tên]` (50 giá trị mỗi phương pháp).

In [ ]:
D10, K10 = 5, 4
rng10 = np.random.default_rng(123)
THETA10 = rng10.normal(0, 1, (K10, D10))
logging_weights = THETA10 + rng10.normal(0, 1.0, THETA10.shape)


def click_prob(X):
    return 1 / (1 + np.exp(-(X @ THETA10.T - 1.0 + 2.0 * np.sin(2 * X[:, :1]) * np.array([1, -1, 1, -1]))))


def logging_policy_probs(X):
    z = 0.5 * X @ logging_weights.T
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def new_policy(X):
    return (X @ THETA10.T).argmax(1)


X_big = np.random.default_rng(7).normal(size=(200_000, D10))
true_value = click_prob(X_big)[np.arange(len(X_big)), new_policy(X_big)].mean()
logging_value = (click_prob(X_big) * logging_policy_probs(X_big)).sum(1).mean()
print(f"giá trị thật: chính sách log = {logging_value:.4f} | chính sách mới = {true_value:.4f}")

estimates10 = {"naive": [], "IPS": [], "SNIPS": []}
for trial in range(50):
    r = np.random.default_rng(trial)
    X = r.normal(size=(3000, D10))
    probs = logging_policy_probs(X)
    A = np.array([r.choice(K10, p=p) for p in probs])
    R = (r.random(3000) < click_prob(X)[np.arange(3000), A]).astype(float)
    ### START CODE HERE ### (≈ 5 dòng)
    # w = 1[A == new_policy(X)] / propensity; naive, IPS, SNIPS
    pass
    ### END CODE HERE ###
for k, v in estimates10.items():
    print(f"{k:6s}: trung bình = {np.mean(v):.4f} | thiên lệch = {np.mean(v) - true_value:+.4f} | độ lệch chuẩn = {np.std(v):.4f}")

In [ ]:
check_greater(abs(np.mean(estimates10["naive"]) - true_value), 0.02, "naive bị thiên lệch")
check_less(abs(np.mean(estimates10["IPS"]) - true_value), 0.015, "IPS gần như không thiên lệch")
check_less(np.std(estimates10["SNIPS"]), np.std(estimates10["IPS"]), "SNIPS phương sai nhỏ hơn IPS")